# Pratica lezione 2. Bandit multi-braccio: UCB ed epsilon-greedy

**DIH DATAlife — Corso di Apprendimento per Rinforzo da zero**

In questa pratica lavorerai sul problema del bandit multi-braccio che hai visto nella lezione teorica (`clase02_bandidos.ipynb`). Dovrai completare due esercizi:

1. **Implementare UCB (Upper Confidence Bound) da zero**, completando la formula del bonus di incertezza e la selezione dell'azione
2. **Confrontare l'effetto di diversi valori di ε** nell'epsilon-greedy sullo stesso bandit

Ogni esercizio contiene:

- Una consegna con suggerimenti
- Una cella di codice con la struttura di partenza e commenti `# TODO: ...` dove devi completare tu
- Una soluzione nascosta, da aprire se ti blocchi, cliccando per espanderla

**Prova a risolvere ogni esercizio da solo prima di guardare la soluzione.**

## Preparazione: il bandit di partenza

> Esegui prima queste celle. Definiscono l'ambiente del bandit che userete nei due esercizi, in modo che tutti partiate dalla stessa base.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fijamos una semilla para que el experimento sea reproducible
np.random.seed(42)

n_brazos = 10
n_pasos = 1000


In [ ]:
class Bandido:
    """Bandido multi-brazo: cada brazo tiene un valor real fijo y desconocido
    para el agente. Al tirar de un brazo, se recibe una recompensa con ruido
    alrededor de ese valor real (como en el capítulo de teoría)."""

    def __init__(self, valores_reales):
        self.valores_reales = valores_reales
        self.n_brazos = len(valores_reales)

    def tirar_de_la_palanca(self, brazo):
        return np.random.normal(loc=self.valores_reales[brazo], scale=1.0)


# Creamos un bandido de ejemplo con valores reales aleatorios
valores_reales = np.random.normal(loc=0.0, scale=1.0, size=n_brazos)
bandido = Bandido(valores_reales)
print("Bandido creado con", bandido.n_brazos, "brazos")


---

## Esercizio uno. Implementa UCB da zero

Ricorda l'idea di UCB della lezione teorica: invece di guardare solo la tua stima attuale del valore di ciascuna azione, le sommi un **bonus di incertezza**. Questo bonus è grande per i bracci poco provati e piccolo per quelli che conosci già bene. A ogni passo scegli il braccio la cui somma **valore stimato + bonus** è più alta.

La formula del bonus, per il braccio *a* al passo *t*, è:

$$\text{bono}(a) = c \cdot \sqrt{\frac{\ln(t)}{N(a)}}$$

dove `N(a)` è il numero di volte in cui hai provato il braccio *a*, e `c` è una costante che controlla quanto pesa l'esplorazione.

### Suggerimenti

- Il bonus cresce con `log(t)`: man mano che il tempo avanza, "conviene" continuare a controllare i bracci poco provati
- Il bonus decresce con la radice quadrata di `N(a)`: più volte provi un braccio, meno incertezza hai sul suo valore
- Usa `np.log(t)` e `np.sqrt(...)` di `numpy`
- Per scegliere il braccio con la somma più alta, usa `np.argmax(valores_estimados + bono)`
- La parte "provare ogni braccio almeno una volta" è già risolta nella struttura di partenza: completala solo se te la chiede il `# TODO`

**Completa i due `# TODO` della funzione `ejecutar_ucb`.**

In [ ]:
def ejecutar_ucb(bandido, c, n_pasos):
    """Estrategia UCB: combina el valor estimado con un bono de incertidumbre
    que favorece a los brazos poco probados."""
    valores_estimados = np.zeros(bandido.n_brazos)
    conteo_intentos = np.zeros(bandido.n_brazos)
    recompensas = np.zeros(n_pasos)

    for t in range(1, n_pasos + 1):
        # Si algún brazo aún no se ha probado, se elige primero (bono "infinito")
        brazos_sin_probar = np.where(conteo_intentos == 0)[0]
        if len(brazos_sin_probar) > 0:
            brazo = brazos_sin_probar[0]
        else:
            # TODO: calcula el bono de incertidumbre para cada brazo
            # bono(a) = c * sqrt(log(t) / N(a))
            bono = None

            # TODO: elige el brazo cuya suma "valor estimado + bono" sea la más alta
            brazo = None

        recompensa = bandido.tirar_de_la_palanca(brazo)
        conteo_intentos[brazo] += 1

        # Actualización incremental de la media muestral (igual que en la clase de teoría)
        paso = 1.0 / conteo_intentos[brazo]
        valores_estimados[brazo] += paso * (recompensa - valores_estimados[brazo])

        recompensas[t - 1] = recompensa

    return recompensas


# Ejecuta tu implementación con c = 2.0
recompensas_ucb = ejecutar_ucb(bandido, c=2.0, n_pasos=n_pasos)
recompensa_media_acumulada = np.cumsum(recompensas_ucb) / np.arange(1, n_pasos + 1)

plt.figure(figsize=(8, 4))
plt.plot(recompensa_media_acumulada)
plt.xlabel("Paso de tiempo")
plt.ylabel("Recompensa media acumulada")
plt.title("Tu implementación de UCB")
plt.grid(alpha=0.3)
plt.show()

print("Recompensa media acumulada final:", round(recompensa_media_acumulada[-1], 3))


<details>
<summary>💡 Vedi la soluzione</summary>

```python
def ejecutar_ucb(bandido, c, n_pasos):
    """Strategia UCB: combina il valore stimato con un bonus di incertezza
    che favorisce i bracci poco provati."""
    valores_estimados = np.zeros(bandido.n_brazos)
    conteo_intentos = np.zeros(bandido.n_brazos)
    recompensas = np.zeros(n_pasos)

    for t in range(1, n_pasos + 1):
        # Se un braccio non è ancora stato provato, lo scegliamo per primo (bonus "infinito")
        brazos_sin_probar = np.where(conteo_intentos == 0)[0]
        if len(brazos_sin_probar) > 0:
            brazo = brazos_sin_probar[0]
        else:
            # Bonus di incertezza: cresce con log(t), decresce con la radice
            # quadrata del numero di volte in cui è stato provato ciascun braccio
            bono = c * np.sqrt(np.log(t) / conteo_intentos)

            # Scegliamo il braccio con il "valore stimato + bonus" più alto
            brazo = np.argmax(valores_estimados + bono)

        recompensa = bandido.tirar_de_la_palanca(brazo)
        conteo_intentos[brazo] += 1

        paso = 1.0 / conteo_intentos[brazo]
        valores_estimados[brazo] += paso * (recompensa - valores_estimados[brazo])

        recompensas[t - 1] = recompensa

    return recompensas


recompensas_ucb = ejecutar_ucb(bandido, c=2.0, n_pasos=n_pasos)
recompensa_media_acumulada = np.cumsum(recompensas_ucb) / np.arange(1, n_pasos + 1)
print("Recompensa media accumulata finale:", round(recompensa_media_acumulada[-1], 3))
```
</details>

---

## Esercizio due. Confronta diversi valori di epsilon

Nella lezione teorica hai visto che ε controlla quanto esplora l'epsilon-greedy: un ε piccolo sfrutta quasi sempre, mentre un ε grande esplora molto di più. Ora lo verificherai tu stesso.

Addestrerai l'epsilon-greedy tre volte sullo stesso bandit, una per ciascun valore della lista `[0.01, 0.1, 0.3]`, e disegnerai le tre curve di ricompensa media accumulata in un unico grafico per confrontarle.

### Suggerimenti

- La funzione `ejecutar_epsilon_greedy` è già completa: non devi modificarla
- Per ogni `epsilon` della lista, ripeti l'esperimento `n_repeticiones` volte e calcola la media, come nella lezione teorica, per attenuare il caso
- Non dimenticare di rigenerare `valores_reales` a ogni ripetizione (così la media non dipende da un'unica configurazione di bracci)
- Usa `plt.plot(curva, label=f"epsilon = {epsilon}")` dentro il ciclo per aggiungere ogni curva allo stesso grafico
- Alla fine, non dimenticare `plt.legend()` per distinguere le tre curve

**Completa i `# TODO` del ciclo.**

In [ ]:
def ejecutar_epsilon_greedy(bandido, epsilon, n_pasos):
    """Estrategia epsilon-greedy: explora al azar con probabilidad epsilon,
    y el resto de las veces elige la acción con mejor valor estimado."""
    valores_estimados = np.zeros(bandido.n_brazos)
    conteo_intentos = np.zeros(bandido.n_brazos)
    recompensas = np.zeros(n_pasos)

    for t in range(n_pasos):
        if np.random.random() < epsilon:
            brazo = np.random.randint(bandido.n_brazos)
        else:
            brazo = np.argmax(valores_estimados)

        recompensa = bandido.tirar_de_la_palanca(brazo)
        conteo_intentos[brazo] += 1
        paso = 1.0 / conteo_intentos[brazo]
        valores_estimados[brazo] += paso * (recompensa - valores_estimados[brazo])
        recompensas[t] = recompensa

    return recompensas


valores_epsilon = [0.01, 0.1, 0.3]
n_repeticiones = 100

plt.figure(figsize=(10, 5))

# TODO: recorre "valores_epsilon" con un bucle for
for epsilon in valores_epsilon:
    recompensas_totales = np.zeros(n_pasos)

    # TODO: repite el experimento "n_repeticiones" veces y suma los resultados
    for _ in range(n_repeticiones):
        # TODO: regenera "valores_reales" al azar en cada repetición
        valores_reales_rep = None

        # TODO: crea un Bandido con esos valores reales
        bandido_rep = None

        # TODO: acumula las recompensas de ejecutar_epsilon_greedy en recompensas_totales
        pass

    recompensas_totales /= n_repeticiones

    # TODO: calcula la recompensa media acumulada a partir de "recompensas_totales"
    curva = None

    plt.plot(curva, label=f"epsilon = {epsilon}")

plt.xlabel("Paso de tiempo")
plt.ylabel("Recompensa media acumulada")
plt.title("Comparación de valores de epsilon en epsilon-greedy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


<details>
<summary>💡 Vedi la soluzione</summary>

```python
valores_epsilon = [0.01, 0.1, 0.3]
n_repeticiones = 100

plt.figure(figsize=(10, 5))

for epsilon in valores_epsilon:
    recompensas_totales = np.zeros(n_pasos)

    for _ in range(n_repeticiones):
        # Rigeneriamo i valori reali a ogni ripetizione, così che la media
        # non dipenda da un'unica configurazione di bracci
        valores_reales_rep = np.random.normal(loc=0.0, scale=1.0, size=n_brazos)
        bandido_rep = Bandido(valores_reales_rep)
        recompensas_totales += ejecutar_epsilon_greedy(bandido_rep, epsilon, n_pasos)

    recompensas_totales /= n_repeticiones
    curva = np.cumsum(recompensas_totales) / np.arange(1, n_pasos + 1)
    plt.plot(curva, label=f"epsilon = {epsilon}")

plt.xlabel("Passo temporale")
plt.ylabel("Ricompensa media accumulata")
plt.title("Confronto dei valori di epsilon nell'epsilon-greedy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
```

**Cosa dovresti osservare:** con ε = 0,01 si esplora quasi per niente, quindi l'apprendimento è lento all'inizio. Con ε = 0,3 si esplora così tanto che la curva resta più bassa anche nel lungo periodo, perché si sprecano ancora molti tentativi sui bracci cattivi. Il valore intermedio, ε = 0,1, di solito trova il miglior equilibrio in questo esperimento.
</details>

---

## Per continuare a esercitarti

Se ti è avanzato del tempo, riprendi le idee di "Mettilo in pratica" del manuale:

- **Cambia il numero di bracci** `n_brazos` (per esempio, a 5 o a 20) e verifica se UCB mantiene il suo vantaggio sull'epsilon-greedy
- **Prova valori diversi di `c`** nel tuo UCB e annota da quale valore in poi la strategia inizia a esplorare troppo

**Condividi con il tuo team quale combinazione di parametri ha trovato prima il braccio migliore!**